# 6.5.4 Mechanistic Interpretability

Analyse attention head patterns and implement the logit lens to inspect model internals.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
T, d, dk, H, V = 8, 32, 8, 4, 50
tokens = rng.integers(0, V, T)
E = rng.standard_normal((V, d))
residual = E[tokens]

def softmax(x): e=np.exp(x-x.max(-1,keepdims=True)); return e/e.sum(-1,keepdims=True)
def attn(Q,K,V, mask=None):
    s = Q@K.T/np.sqrt(dk)
    if mask is not None: s = np.where(mask, s, -1e9)
    w = softmax(s); return w@V, w

mask = np.tril(np.ones((T,T), bool))
head_ws = []
for h in range(H):
    Wq=rng.standard_normal((d,dk))*0.1; Wk=rng.standard_normal((d,dk))*0.1; Wv=rng.standard_normal((d,dk))*0.1
    _,w = attn(residual@Wq, residual@Wk, residual@Wv, mask)
    head_ws.append(w)
    ent = (-w*np.log(w+1e-10)).sum(-1).mean()
    print(f'Head {h}: mean attention entropy = {ent:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im = axes[0].imshow(head_ws[0], cmap='Blues', vmin=0, vmax=1)
axes[0].set(title='Head 0 Attention Pattern', xlabel='Key', ylabel='Query')
plt.colorbar(im, ax=axes[0])

entropies = [(-w*np.log(w+1e-10)).sum(-1).mean() for w in head_ws]
axes[1].bar(range(H), entropies, color='steelblue')
axes[1].set(xlabel='Head', ylabel='Entropy', title='Attention Entropy per Head')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output/mechanistic_interp.png')
print('Saved mechanistic_interp.png')